In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import glob

# 配置
DATA_DIR = "/home/youtian/NCBI_ten_genomes"
OUT_BASE = "spacedust_batch_output_new"
TARGET_DIR = "my_project/data/KEGG_70"
os.makedirs(OUT_BASE, exist_ok=True)

# 获取所有 faa 文件
faa_files = glob.glob(os.path.join(DATA_DIR, "*.faa"))
print(f"找到 {len(faa_files)} 个 faa 文件:")
for f in faa_files:
    print(f"  - {f}")

找到 10 个 faa 文件:
  - /home/youtian/NCBI_ten_genomes/GCF_000024265.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000011865.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000016965.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000009045.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000346065.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000009205.2_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000005845.2_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000022165.1_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000195955.2_spacedust.faa
  - /home/youtian/NCBI_ten_genomes/GCF_000012525.1_spacedust.faa


In [2]:
def run_command(cmd):
    """运行 shell 命令"""
    print(f"[Running]: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

In [3]:
def run_spacedust_pipeline(query_faa, out_dir):
    """对单个 faa 文件运行 Spacedust 标准流程"""
    os.makedirs(out_dir, exist_ok=True)
    
    # 1.1 创建 Query DB
    qdb = f"{out_dir}/querydb"
    if not os.path.exists(qdb):
        run_command(f"spacedust createsetdb {query_faa} {qdb} {out_dir}/tmp_q")

    # 1.2 Target DB (已存在)
    tdb = f"{TARGET_DIR}/keggclusterdb"

    # 1.3 运行搜索与聚类
    resdb = f"{out_dir}/resultdb"
    if not os.path.exists(resdb):
        run_command(f"spacedust clustersearch {qdb} {tdb} {resdb} {out_dir}/tmp_res --threads 8")

    # 1.4 格式化输出
    if not os.path.exists(f"{resdb}_pref"):
        run_command(f"spacedust prefixid {out_dir}/tmp_res/latest/clusters {resdb}_pref --tsv 1")
    if not os.path.exists(f"{resdb}_h_pref"):
        run_command(f"spacedust prefixid {out_dir}/tmp_res/latest/clusters_h {resdb}_h_pref --tsv 1")
    
    print(f"Pipeline completed for {query_faa}")

In [4]:
def read_lookup(lookup_file):
    mapping = pd.read_csv(lookup_file, sep="\t", header=None,
                          names=['seqid','header','setid'],
                          dtype = {'seqid': 'int', 'header': 'str', 'setid': 'int'})
    mapping[['acc', 'idx', 'start', 'end']] = mapping['header'].str.rsplit("_", n=3, expand=True)
    mapping = mapping.astype({'idx': 'int', 'start': 'int', 'end': 'int'})
    mapping['idx0'] = mapping['idx'] + 1
    return mapping

def query_read_lookup(lookup_file):
    mapping = pd.read_csv(lookup_file, sep="\t", header=None,
                          names=['seqid','header','setid'],
                          dtype = {'seqid': 'int', 'header': 'str', 'setid': 'int'})
    mapping[['acc','idx0','idx','start','end']] = mapping['header'].str.rsplit("_", n = 4, expand = True)
    mapping = mapping.astype({'idx0':'int','idx':'int','start':'int','end':'int'})
    return mapping

def separate_clustersearch_lookup(lookup_file):
    mapping = query_read_lookup(lookup_file)
    mapping.sort_values('idx')
    mapping['strand'] = np.where(mapping["start"] < mapping["end"], True, False) 
    mapping['accnext'] = mapping.acc.shift(-1)
    n_query_genomes = mapping['setid'].nunique()
    qs_mapping = {}
    qs_cssp = {}
    for i in range(n_query_genomes):
        qs_mapping[i] = mapping[mapping['setid'] == i]
        qs_mapping[i]['strandnext'] = qs_mapping[i].strand.shift(-1)
        qs_mapping[i]['iscssp'] = (qs_mapping[i]['strand'] == qs_mapping[i]['strandnext'])
        qs_mapping[i]['isacc'] = (qs_mapping[i]['acc']==qs_mapping[i]['accnext'])
        df_cssp = qs_mapping[i][(qs_mapping[i]['iscssp']==True)&(qs_mapping[i]['isacc']==True)] 
        qs_cssp[i] = df_cssp['idx'].reset_index(drop=True)        
    return (n_query_genomes, qs_mapping, qs_cssp)

def process_clusterresult(clusterresult_dir):
    clusterresult_file = f"{clusterresult_dir}/resultdb_pref"
    clusterheaderresult_file = f"{clusterresult_dir}/resultdb_h_pref"
    clusterresult = pd.read_csv(clusterresult_file, sep='\t', header=None)[[0,1,2]]
    clusterresult = clusterresult.rename(columns={0:'clusterid',1:'qid',2:'tid'})
    clusterresult = clusterresult.astype({'clusterid':'int', 'qid':'int', 'tid':'int'})
    clusterresult = clusterresult.sort_values(['clusterid']).reset_index(drop=True)
    g = clusterresult.groupby('clusterid').qid
    clusterresult_agg = pd.concat([g.apply(list)], axis=1, keys=['genes'])
    clusterresult_agg['clusterid'] = clusterresult_agg.index
    clusterheaderresult = pd.read_csv(clusterheaderresult_file, sep="\t", header=None, names=['clusterid','qsetid','tsetid','clustp','orderp','num'])
    clusterheaderresult = clusterheaderresult.sort_values('clusterid').reset_index(drop=True)
    col_dict = {0:'clusterid',1:'qid',2:'tid'}
    clusterresult_dict = {}
    for i in clusterresult.index:
        cluster = clusterresult.iat[i,0]
        qid = clusterresult.iat[i,1] 
        tid = clusterresult.iat[i,2] 
        if cluster not in clusterresult_dict.keys():
            clusterresult_dict[cluster] = {}
        clusterresult_dict[cluster][qid] = tid
    return (clusterresult_agg, clusterresult_dict, clusterheaderresult)

In [5]:
def get_hits_and_tid(clusterresult_dir, query_lookup, target_lookup):
    n_query_genomes, qs_mapping, qs_cssp = separate_clustersearch_lookup(query_lookup)
    df_target_lookup = read_lookup(target_lookup)
    clusterresult_agg, clusterresult_dict, clusterheaderresult = process_clusterresult(clusterresult_dir)
    qs_clusterheaderresult = {}
    qs_matrix = {}
    tid_1_matrix = {}
    tid_2_matrix = {}
    qs_mat_df = {}
    tid_1_mat_df = {}
    tid_2_mat_df = {}
    for k in range(n_query_genomes):
        result_df = clusterheaderresult[clusterheaderresult['qsetid'] == k]
        mapping = qs_mapping[k]
        pairs_query = len(mapping['seqid']) - 1
        n_reference = result_df['tsetid'].max() + 1
        dimensions = (pairs_query, n_reference)
        qs_matrix[k] = np.zeros(dimensions)
        tid_1_matrix[k] = np.zeros(dimensions)
        tid_2_matrix[k] = np.zeros(dimensions)
        for cluster in clusterresult_agg['clusterid']:
            j = result_df['tsetid'][cluster]
            for i in clusterresult_agg['genes'][cluster]:
                if (i+1) in clusterresult_agg['genes'][cluster]:
                    qs_matrix[k][i][j] = 1
                    tseqid_1 = clusterresult_dict[cluster][i]
                    tid_1 = df_target_lookup.at[tseqid_1,'idx']
                    tid_1_matrix[k][i][j] = tid_1
                    tseqid_2 = clusterresult_dict[cluster][i+1]
                    tid_2 = df_target_lookup.at[tseqid_2,'idx']
                    tid_2_matrix[k][i][j] = tid_2
        idx_query = mapping['idx'][:-1].values.flatten().reshape(1, pairs_query)  
        qs_mat = np.append(idx_query, qs_matrix[k].T, axis=0).astype(int)
        qs_mat_df[k] = pd.DataFrame(data=qs_mat[1:,:],columns=qs_mat[0,:])
        qs_mat_df[k] = qs_mat_df[k][qs_mat_df[k].columns.intersection(qs_cssp[k])]
        tid_1_mat = np.append(idx_query, tid_1_matrix[k].T, axis=0).astype(int)    
        tid_1_mat_df[k] = pd.DataFrame(data=tid_1_mat[1:,:],columns=tid_1_mat[0,:])
        tid_1_mat_df[k] = tid_1_mat_df[k][tid_1_mat_df[k].columns.intersection(qs_cssp[k])]
        tid_2_mat = np.append(idx_query, tid_2_matrix[k].T, axis=0).astype(int)     
        tid_2_mat_df[k] = pd.DataFrame(data=tid_2_mat[1:,:],columns=tid_2_mat[0,:])
        tid_2_mat_df[k] = tid_2_mat_df[k][tid_2_mat_df[k].columns.intersection(qs_cssp[k])]  
    return (qs_mat_df, tid_1_mat_df, tid_2_mat_df)

def scale_conservation_matrix(qs_mat_df, lookup_file):
    qs_mat_df_centralized = {}
    keys = qs_mat_df.keys()
    n_query_genomes, qs_mapping, qs_cssp = separate_clustersearch_lookup(lookup_file)
    for k in keys:
        qs_mat_df_centralized[k] = qs_mat_df[k].subtract(qs_mat_df[k].sum(axis=1)/len(qs_cssp[k]), axis=0)
    return qs_mat_df_centralized

In [ ]:
def process_single_genome(faa_file, out_base, target_dir):
    """处理单个基因组 faa 文件，生成 conservation matrix CSV"""
    genome_name = os.path.basename(faa_file).replace('.faa', '')
    out_dir = os.path.join(out_base, genome_name)
    
    print(f"\n{'='*50}")
    print(f"处理基因组：{genome_name}")
    print(f"{'='*50}")
    
    # 运行 spacedust pipeline
    run_spacedust_pipeline(faa_file, out_dir)
    
    # 生成 conservation matrix
    query_lookup = f"{out_dir}/querydb.lookup"
    target_lookup = f"{target_dir}/keggclusterdb.lookup"
    
    qs_mat_df, tid_1_mat_df, tid_2_mat_df = get_hits_and_tid(out_dir, query_lookup, target_lookup)
    nodst_mat_df = scale_conservation_matrix(qs_mat_df, query_lookup)
    
    # 保存 CSV
    csv_path = f"{out_dir}/nodst_conservation_matrix.csv"
    original_csv_path = f"{out_dir}/original_conservation_matrix.csv"
    if len(nodst_mat_df) > 0:
        nodst_mat_df[0].to_csv(csv_path)
        qs_mat_df[0].to_csv(original_csv_path)
        print(f"已保存 CSV: {csv_path}")
    else:
        print(f"警告：{genome_name} 没有生成有效的 matrix")
    
    return csv_path

In [7]:
# 批量处理所有 10 个 faa 文件
target_dir = TARGET_DIR
results = []

for faa_file in faa_files:
    try:
        csv_path = process_single_genome(faa_file, OUT_BASE, target_dir)
        results.append({'genome': os.path.basename(faa_file), 'csv': csv_path, 'status': 'success'})
    except Exception as e:
        results.append({'genome': os.path.basename(faa_file), 'csv': None, 'status': f'error: {str(e)}'})
        print(f"处理 {faa_file} 时出错：{e}")

# 输出汇总
print("\n" + "="*50)
print("批量处理完成!")
print("="*50)
results_df = pd.DataFrame(results)
print(results_df)


处理基因组：GCF_000024265.1_spacedust
[Running]: spacedust createsetdb /home/youtian/NCBI_ten_genomes/GCF_000024265.1_spacedust.faa spacedust_batch_output_new/GCF_000024265.1_spacedust/querydb spacedust_batch_output_new/GCF_000024265.1_spacedust/tmp_q
Create directory spacedust_batch_output_new/GCF_000024265.1_spacedust/tmp_q
createsetdb /home/youtian/NCBI_ten_genomes/GCF_000024265.1_spacedust.faa spacedust_batch_output_new/GCF_000024265.1_spacedust/querydb spacedust_batch_output_new/GCF_000024265.1_spacedust/tmp_q 

MMseqs Version:          	2.e56c505
Database type            	0
Shuffle input database   	false
Createdb mode            	0
Write lookup file        	1
Offset of numeric ids    	0
Compressed               	0
Verbosity                	3
Min codons in orf        	30
Max codons in length     	32734
Max orf gaps             	2147483647
Contig start mode        	2
Contig end mode          	2
Orf start mode           	1
Forward frames           	1,2,3
Reverse frames           	1,2,3
